# AI-CFD DGCNN Training — Colab Complete Resume Version

이 노트북은 런타임 초기화에 대비해 처음부터 끝까지 다시 정리한 완성본이다.

Google Drive에 아래 파일이 있어야 한다.

- `MyDrive/ai-cfd-flow-prediction/data/05_cfd_csv.zip`
- `MyDrive/ai-cfd-flow-prediction/data/fps_indices_7000.npz`

학습 결과는 자동으로 아래에 저장된다.

- `MyDrive/ai-cfd-flow-prediction/dgcnn/best_model.pt`
- `MyDrive/ai-cfd-flow-prediction/dgcnn/scalers.npz`
- `MyDrive/ai-cfd-flow-prediction/dgcnn/last_checkpoint.pt`

런타임이 초기화되면 Cell 1 → 2 → 3 → 5 순서로 다시 실행하면 된다.
Cell 5는 `last_checkpoint.pt`가 있으면 자동 resume하고, 없으면 새 학습을 시작한다.


## Cell 1 — GPU 확인


In [1]:
!nvidia-smi

import torch

print()
print("PyTorch        :", torch.__version__)
print("CUDA available :", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU가 잡히지 않았습니다. "
        "Colab에서 런타임 유형을 T4 GPU로 변경하세요."
    )

print("GPU            :", torch.cuda.get_device_name(0))
print(
    "GPU memory     :",
    torch.cuda.get_device_properties(0).total_memory / 1024**3,
    "GB",
)


Thu Aug 20 22:16:32 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Cell 2 — Google Drive 연결 및 영구 경로 설정


In [2]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

DRIVE_ROOT = Path(
    "/content/drive/MyDrive/ai-cfd-flow-prediction"
)

DRIVE_DATA = DRIVE_ROOT / "data"
DRIVE_DGCNN = DRIVE_ROOT / "dgcnn"

DRIVE_ZIP = DRIVE_DATA / "05_cfd_csv.zip"
DRIVE_FPS = DRIVE_DATA / "fps_indices_7000.npz"

BEST_MODEL = DRIVE_DGCNN / "best_model.pt"
SCALER = DRIVE_DGCNN / "scalers.npz"
LAST_CHECKPOINT = DRIVE_DGCNN / "last_checkpoint.pt"
TEST_LOG = DRIVE_DGCNN / "test_evaluation.txt"

DRIVE_DGCNN.mkdir(
    parents=True,
    exist_ok=True,
)

required_files = [
    DRIVE_ZIP,
    DRIVE_FPS,
]

for path in required_files:
    if not path.exists():
        raise FileNotFoundError(
            f"Google Drive 파일을 찾을 수 없습니다:\n{path}"
        )

print("=" * 78)
print("GOOGLE DRIVE READY")
print("=" * 78)
print("CSV ZIP         :", DRIVE_ZIP)
print("FPS index       :", DRIVE_FPS)
print("DGCNN output    :", DRIVE_DGCNN)
print("Best model      :", BEST_MODEL)
print("Scaler          :", SCALER)
print("Last checkpoint :", LAST_CHECKPOINT)
print("=" * 78)


Mounted at /content/drive
GOOGLE DRIVE READY
CSV ZIP         : /content/drive/MyDrive/ai-cfd-flow-prediction/data/05_cfd_csv.zip
FPS index       : /content/drive/MyDrive/ai-cfd-flow-prediction/data/fps_indices_7000.npz
DGCNN output    : /content/drive/MyDrive/ai-cfd-flow-prediction/dgcnn
Best model      : /content/drive/MyDrive/ai-cfd-flow-prediction/dgcnn/best_model.pt
Scaler          : /content/drive/MyDrive/ai-cfd-flow-prediction/dgcnn/scalers.npz
Last checkpoint : /content/drive/MyDrive/ai-cfd-flow-prediction/dgcnn/last_checkpoint.pt


## Cell 3 — GitHub 코드 및 학습 데이터 자동 복구


In [3]:
import shutil
import subprocess
from pathlib import Path

REPO = Path(
    "/content/ai-cfd-flow-prediction"
)

DATA_ROOT = Path(
    "/content/ai-cfd-data"
)

CSV_DIR = (
    DATA_ROOT
    / "05_cfd_csv"
)

FPS_DST = (
    REPO
    / "04_cfd_dataset"
    / "fps_indices_7000.npz"
)

# ================================================================
# 1. GitHub repository
# ================================================================

if REPO.exists():

    print(
        "[Git] Existing repository found -> pull latest main",
        flush=True,
    )

    subprocess.run(
        [
            "git",
            "-C",
            str(REPO),
            "pull",
            "origin",
            "main",
        ],
        check=True,
    )

else:

    print(
        "[Git] Repository not found -> clone",
        flush=True,
    )

    subprocess.run(
        [
            "git",
            "clone",
            "https://github.com/hehong01/ai-cfd-flow-prediction.git",
            str(REPO),
        ],
        check=True,
    )

# ================================================================
# 2. CFD CSV restore
# ================================================================

def count_direct_csv():
    if not CSV_DIR.exists():
        return 0

    return len(
        list(
            CSV_DIR.glob("*.csv")
        )
    )

csv_count = count_direct_csv()

if csv_count != 300:

    print(
        f"[Data] Local CSV count = {csv_count} -> restore from Drive",
        flush=True,
    )

    shutil.rmtree(
        DATA_ROOT,
        ignore_errors=True,
    )

    DATA_ROOT.mkdir(
        parents=True,
        exist_ok=True,
    )

    local_zip = Path(
        "/content/05_cfd_csv.zip"
    )

    shutil.copy2(
        DRIVE_ZIP,
        local_zip,
    )

    # ZIP 내부에 05_cfd_csv/ 폴더가 있으므로
    # /content/ai-cfd-data 에 직접 압축 해제한다.
    subprocess.run(
        [
            "unzip",
            "-q",
            "-o",
            str(local_zip),
            "-d",
            str(DATA_ROOT),
        ],
        check=True,
    )

    csv_count = count_direct_csv()

else:

    print(
        "[Data] Existing 300 local CSV files -> reuse",
        flush=True,
    )

if csv_count != 300:
    raise RuntimeError(
        "CFD CSV 복구 실패:\n"
        f"Expected 300, found {csv_count}\n"
        f"Directory: {CSV_DIR}"
    )

# ================================================================
# 3. FPS index restore
# ================================================================

FPS_DST.parent.mkdir(
    parents=True,
    exist_ok=True,
)

shutil.copy2(
    DRIVE_FPS,
    FPS_DST,
)

if not FPS_DST.exists():
    raise RuntimeError(
        f"FPS 복사 실패:\n{FPS_DST}"
    )

# ================================================================
# 4. State check
# ================================================================

commit = (
    subprocess.check_output(
        [
            "git",
            "-C",
            str(REPO),
            "rev-parse",
            "--short",
            "HEAD",
        ],
        text=True,
    )
    .strip()
)

print()
print("=" * 78)
print("COLAB ENVIRONMENT READY")
print("=" * 78)
print("Git commit       :", commit)
print("CSV count        :", csv_count)
print("CSV directory    :", CSV_DIR)
print("FPS exists       :", FPS_DST.exists())
print("FPS path         :", FPS_DST)
print(
    "Checkpoint exists:",
    LAST_CHECKPOINT.exists(),
)
print("=" * 78)


[Git] Repository not found -> clone
[Data] Local CSV count = 0 -> restore from Drive

COLAB ENVIRONMENT READY
Git commit       : 7bb08b3
CSV count        : 300
CSV directory    : /content/ai-cfd-data/05_cfd_csv
FPS exists       : True
FPS path         : /content/ai-cfd-flow-prediction/04_cfd_dataset/fps_indices_7000.npz
Checkpoint exists: True


## Cell 4 — Dataset 전체 검증

처음 세팅하거나 데이터 경로가 의심될 때 실행한다.

정상이라면 다음이 모두 확인되어야 한다.

- TRAIN: 240 samples
- VAL: 30 samples
- TEST: 30 samples
- TOTAL: 300 samples
- 7000 points/sample


In [4]:
%cd /content/ai-cfd-flow-prediction
!python -u 04_cfd_dataset/dataset.py


/content/ai-cfd-flow-prediction
DGCNN DATASET FULL TEST

[TRAIN]
samples : 240
   25/240  face_0009_05mps OK
   50/240  face_0017_08mps OK
   75/240  face_0025_10mps OK
  100/240  face_0034_05mps OK
  125/240  face_0042_08mps OK
  150/240  face_0050_10mps OK
  175/240  face_0059_05mps OK
  200/240  face_0067_08mps OK
  225/240  face_0075_10mps OK
  240/240  face_0080_10mps OK
TRAIN PASSED: 240 samples

[VAL]
samples : 30
   25/30  face_0089_05mps OK
   30/30  face_0090_10mps OK
VAL PASSED: 30 samples

[TEST]
samples : 30
   25/30  face_0099_05mps OK
   30/30  face_0100_10mps OK
TEST PASSED: 30 samples

FULL DATASET TEST PASSED
Total samples checked : 300
Points / sample       : 7000
Input                : [x, y, z, velocity]
Input shape          : (7000, 4)
Target               : [HTC, wall_shear]
Target shape         : (7000, 2)
Tensor dtype         : torch.float32


## Cell 5 — DGCNN 본학습 / 자동 Resume / 실시간 로그

이 셀은 완성본이다.

- `last_checkpoint.pt`가 없으면 새 학습 시작
- 있으면 저장된 다음 epoch부터 자동 resume
- `train.py`의 stdout/stderr를 한 줄씩 직접 읽어서 Colab에 실시간 출력
- 매 epoch 종료 후 `last_checkpoint.pt`가 Google Drive에 갱신
- best validation model은 `best_model.pt`에 유지


In [5]:
# ================================================================
# Cell 5 — DGCNN full training / automatic resume / live logging
# ================================================================

from pathlib import Path
import subprocess
import sys
import torch

# ================================================================
# Paths
# ================================================================

REPO = Path(
    "/content/ai-cfd-flow-prediction"
)

DRIVE_ROOT = Path(
    "/content/drive/MyDrive/ai-cfd-flow-prediction"
)

DRIVE_DGCNN = (
    DRIVE_ROOT
    / "dgcnn"
)

DRIVE_DGCNN.mkdir(
    parents=True,
    exist_ok=True,
)

TRAIN_SCRIPT = (
    REPO
    / "05_model_training"
    / "dgcnn"
    / "train.py"
)

BEST_MODEL = (
    DRIVE_DGCNN
    / "best_model.pt"
)

SCALER = (
    DRIVE_DGCNN
    / "scalers.npz"
)

LAST_CHECKPOINT = (
    DRIVE_DGCNN
    / "last_checkpoint.pt"
)

# ================================================================
# Final training settings
# ================================================================

TARGET_EPOCHS = 100
PATIENCE = 15
BATCH_SIZE = 16

# ================================================================
# Basic checks
# ================================================================

if not torch.cuda.is_available():

    raise RuntimeError(
        "CUDA GPU가 잡히지 않았습니다. "
        "Colab 런타임을 T4 GPU로 변경하세요."
    )

if not TRAIN_SCRIPT.exists():

    raise FileNotFoundError(
        "train.py를 찾을 수 없습니다:\n"
        f"{TRAIN_SCRIPT}"
    )

print(
    "=" * 78,
    flush=True,
)

print(
    "DGCNN FINAL TRAINING LAUNCHER",
    flush=True,
)

print(
    "=" * 78,
    flush=True,
)

print(
    f"GPU             : {torch.cuda.get_device_name(0)}",
    flush=True,
)

print(
    f"Batch size      : {BATCH_SIZE}",
    flush=True,
)

print(
    f"Target epochs   : {TARGET_EPOCHS}",
    flush=True,
)

print(
    f"Patience        : {PATIENCE}",
    flush=True,
)

print(
    f"Best model      : {BEST_MODEL}",
    flush=True,
)

print(
    f"Scaler          : {SCALER}",
    flush=True,
)

print(
    f"Last checkpoint : {LAST_CHECKPOINT}",
    flush=True,
)

print(
    "=" * 78,
    flush=True,
)

print(
    flush=True,
)

# ================================================================
# Fresh training vs resume
# ================================================================

run_training = True
mode = None

if LAST_CHECKPOINT.exists():

    if not SCALER.exists():

        raise RuntimeError(
            "last_checkpoint.pt는 존재하지만 "
            "scalers.npz가 없습니다.\n"
            "Resume할 수 없는 상태입니다."
        )

    print(
        "Existing last_checkpoint.pt detected.",
        flush=True,
    )

    checkpoint = torch.load(
        LAST_CHECKPOINT,
        map_location="cpu",
        weights_only=False,
    )

    required_keys = [
        "epoch",
        "best_epoch",
        "best_val_loss",
        "epochs_without_improvement",
    ]

    for key in required_keys:

        if key not in checkpoint:

            raise KeyError(
                "Resume checkpoint에 필수 항목이 없습니다: "
                f"{key}"
            )

    completed_epoch = int(
        checkpoint[
            "epoch"
        ]
    )

    best_epoch = int(
        checkpoint[
            "best_epoch"
        ]
    )

    best_val_loss = float(
        checkpoint[
            "best_val_loss"
        ]
    )

    no_improve_count = int(
        checkpoint[
            "epochs_without_improvement"
        ]
    )

    print(
        f"Completed epoch  : {completed_epoch}",
        flush=True,
    )

    print(
        f"Best epoch       : {best_epoch}",
        flush=True,
    )

    print(
        f"Best val loss    : {best_val_loss:.8f}",
        flush=True,
    )

    print(
        f"No-improve count : {no_improve_count}",
        flush=True,
    )

    print(
        flush=True,
    )

    if completed_epoch >= TARGET_EPOCHS:

        print(
            "Training is already complete "
            f"({completed_epoch}/{TARGET_EPOCHS}).",
            flush=True,
        )

        run_training = False

    elif no_improve_count >= PATIENCE:

        print(
            "Training already reached the "
            "early-stopping condition.",
            flush=True,
        )

        run_training = False

    else:

        mode = "--resume"

        print(
            "RESUME TRAINING: "
            f"epoch {completed_epoch + 1}부터 시작합니다.",
            flush=True,
        )

else:

    mode = "--overwrite"

    print(
        "No resume checkpoint -> "
        "새 본학습을 시작합니다.",
        flush=True,
    )

print(
    flush=True,
)

# ================================================================
# Launch train.py
# ================================================================

if run_training:

    cmd = [
        sys.executable,
        "-u",
        str(TRAIN_SCRIPT),

        "--batch-size",
        str(BATCH_SIZE),

        "--epochs",
        str(TARGET_EPOCHS),

        "--patience",
        str(PATIENCE),

        "--model-path",
        str(BEST_MODEL),

        "--scaler-path",
        str(SCALER),

        "--checkpoint-path",
        str(LAST_CHECKPOINT),

        mode,
    ]

    print(
        "=" * 78,
        flush=True,
    )

    print(
        "COMMAND",
        flush=True,
    )

    print(
        "=" * 78,
        flush=True,
    )

    print(
        " ".join(cmd),
        flush=True,
    )

    print(
        "=" * 78,
        flush=True,
    )

    print(
        flush=True,
    )

    # ------------------------------------------------------------
    # Start child process.
    #
    # stdout + stderr를 하나의 PIPE로 받고,
    # 한 줄씩 즉시 Colab 출력창에 전달한다.
    # ------------------------------------------------------------

    process = subprocess.Popen(
        cmd,
        cwd=str(REPO),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    if process.stdout is None:

        process.kill()

        raise RuntimeError(
            "train.py stdout pipe를 열 수 없습니다."
        )

    try:

        for line in iter(
            process.stdout.readline,
            "",
        ):

            if line == "":
                break

            print(
                line,
                end="",
                flush=True,
            )

    finally:

        process.stdout.close()

    return_code = process.wait()

    if return_code != 0:

        raise RuntimeError(
            "DGCNN training failed "
            f"(exit code {return_code})."
        )

    print(
        flush=True,
    )

    print(
        "=" * 78,
        flush=True,
    )

    print(
        "DGCNN TRAINING PROCESS FINISHED",
        flush=True,
    )

    print(
        "=" * 78,
        flush=True,
    )


DGCNN FINAL TRAINING LAUNCHER
GPU             : Tesla T4
Batch size      : 16
Target epochs   : 100
Patience        : 15
Best model      : /content/drive/MyDrive/ai-cfd-flow-prediction/dgcnn/best_model.pt
Scaler          : /content/drive/MyDrive/ai-cfd-flow-prediction/dgcnn/scalers.npz
Last checkpoint : /content/drive/MyDrive/ai-cfd-flow-prediction/dgcnn/last_checkpoint.pt

Existing last_checkpoint.pt detected.
Completed epoch  : 26
Best epoch       : 26
Best val loss    : 0.09080548
No-improve count : 0

RESUME TRAINING: epoch 27부터 시작합니다.

COMMAND
/usr/bin/python3 -u /content/ai-cfd-flow-prediction/05_model_training/dgcnn/train.py --batch-size 16 --epochs 100 --patience 15 --model-path /content/drive/MyDrive/ai-cfd-flow-prediction/dgcnn/best_model.pt --scaler-path /content/drive/MyDrive/ai-cfd-flow-prediction/dgcnn/scalers.npz --checkpoint-path /content/drive/MyDrive/ai-cfd-flow-prediction/dgcnn/last_checkpoint.pt --resume

DGCNN CFD TRAINING
Data root       : /content/ai-cfd-data
CSV

## Cell 6 — 최종 held-out TEST 평가

학습이 끝난 뒤 한 번 실행한다.

- test split: face_0091 ~ face_0100
- 30 samples
- full 7000 points/sample
- 결과는 Drive의 `dgcnn/test_evaluation.txt`에도 저장


In [6]:
%cd /content/ai-cfd-flow-prediction

!python -u 05_model_training/dgcnn/evaluate.py \
    --batch-size 16 \
    --model-path "/content/drive/MyDrive/ai-cfd-flow-prediction/dgcnn/best_model.pt" \
    --scaler-path "/content/drive/MyDrive/ai-cfd-flow-prediction/dgcnn/scalers.npz" \
    | tee "/content/drive/MyDrive/ai-cfd-flow-prediction/dgcnn/test_evaluation.txt"


/content/ai-cfd-flow-prediction
DGCNN CFD TEST EVALUATION
Data root       : /content/ai-cfd-data
CSV directory   : /content/ai-cfd-data/05_cfd_csv
Model           : /content/drive/MyDrive/ai-cfd-flow-prediction/dgcnn/best_model.pt
Scaler          : /content/drive/MyDrive/ai-cfd-flow-prediction/dgcnn/scalers.npz
Device          : cuda
GPU             : Tesla T4
PyTorch         : 2.11.0+cu128
Batch size      : 16
Point limit     : None (full FPS point cloud)

CHECKPOINT
Model name      : DGCNNRegressor
Best epoch      : 96
Validation loss : 0.05088102
Input dim       : 4
k               : 20
First kNN graph : raw_xyz
Train kNN chunk : 1024
Eval kNN chunk  : 1024
Train points    : 7000

DGCNN SCALER
CFD SCALER STATISTICS

[INPUT]
x            mean = -0.0006748074785   std =  0.03528007233
y            mean =  0.006994213081   std =  0.05338994376
z            mean = -0.03093699163   std =  0.0467023002
velocity     mean =  7.666666667   std =  2.054804668

[TARGET]
HTC          mean =  53

## Cell 7 — Drive 결과 파일 확인


In [7]:
from pathlib import Path

DRIVE_DGCNN = Path(
    "/content/drive/MyDrive/ai-cfd-flow-prediction/dgcnn"
)

print("=" * 78)
print("DGCNN DRIVE OUTPUTS")
print("=" * 78)

if not DRIVE_DGCNN.exists():

    raise FileNotFoundError(
        f"DGCNN Drive 폴더가 없습니다:\n{DRIVE_DGCNN}"
    )

for path in sorted(
    DRIVE_DGCNN.iterdir()
):

    if path.is_file():

        print(
            f"{path.name:24s} "
            f"{path.stat().st_size / 1024**2:8.2f} MB"
        )

print("=" * 78)


DGCNN DRIVE OUTPUTS
best_model.pt                0.73 MB
last_checkpoint.pt           2.19 MB
scalers.npz                  0.00 MB
test_evaluation.txt          0.00 MB


Cell 8 — 최종 성능 평가


In [8]:
# ================================================================
# FINAL DGCNN RESULT SUMMARY
# ================================================================

from pathlib import Path
import torch

DRIVE_DGCNN = Path(
    "/content/drive/MyDrive/ai-cfd-flow-prediction/dgcnn"
)

BEST_MODEL = DRIVE_DGCNN / "best_model.pt"
LAST_CHECKPOINT = DRIVE_DGCNN / "last_checkpoint.pt"
SCALER = DRIVE_DGCNN / "scalers.npz"
TEST_LOG = DRIVE_DGCNN / "test_evaluation.txt"

print("=" * 78)
print("FINAL DGCNN RESULT SUMMARY")
print("=" * 78)

# ------------------------------------------------
# Best model metadata
# ------------------------------------------------

if not BEST_MODEL.exists():
    raise FileNotFoundError(BEST_MODEL)

best = torch.load(
    BEST_MODEL,
    map_location="cpu",
    weights_only=False,
)

print()
print("[BEST MODEL]")
print("Best epoch       :", best.get("epoch"))
print("Best val loss    :", best.get("val_loss"))
print("Points/sample    :", best.get("point_count"))
print("k                :", best.get("k"))
print("First kNN space  :", best.get("first_knn_space"))

# ------------------------------------------------
# Last training checkpoint
# ------------------------------------------------

if not LAST_CHECKPOINT.exists():
    raise FileNotFoundError(LAST_CHECKPOINT)

last = torch.load(
    LAST_CHECKPOINT,
    map_location="cpu",
    weights_only=False,
)

print()
print("[LAST TRAINING STATE]")
print("Completed epoch  :", last.get("epoch"))
print("Best epoch       :", last.get("best_epoch"))
print("Best val loss    :", last.get("best_val_loss"))
print(
    "No-improve count :",
    last.get("epochs_without_improvement"),
)

# ------------------------------------------------
# File sizes
# ------------------------------------------------

print()
print("[FILES]")

for path in (
    BEST_MODEL,
    LAST_CHECKPOINT,
    SCALER,
    TEST_LOG,
):
    print(
        f"{path.name:24s} "
        f"{path.stat().st_size:,} bytes"
    )

# ------------------------------------------------
# Held-out TEST result
# ------------------------------------------------

print()
print("=" * 78)
print("HELD-OUT TEST RESULT")
print("=" * 78)

if not TEST_LOG.exists():
    raise FileNotFoundError(TEST_LOG)

test_text = TEST_LOG.read_text(
    encoding="utf-8",
    errors="replace",
)

print(test_text)

print("=" * 78)

FINAL DGCNN RESULT SUMMARY

[BEST MODEL]
Best epoch       : 96
Best val loss    : 0.0508810227115949
Points/sample    : 7000
k                : 20
First kNN space  : raw_xyz

[LAST TRAINING STATE]
Completed epoch  : 100
Best epoch       : 96
Best val loss    : 0.0508810227115949
No-improve count : 4

[FILES]
best_model.pt            764,907 bytes
last_checkpoint.pt       2,294,373 bytes
scalers.npz              1,126 bytes
test_evaluation.txt      4,404 bytes

HELD-OUT TEST RESULT
DGCNN CFD TEST EVALUATION
Data root       : /content/ai-cfd-data
CSV directory   : /content/ai-cfd-data/05_cfd_csv
Model           : /content/drive/MyDrive/ai-cfd-flow-prediction/dgcnn/best_model.pt
Scaler          : /content/drive/MyDrive/ai-cfd-flow-prediction/dgcnn/scalers.npz
Device          : cuda
GPU             : Tesla T4
PyTorch         : 2.11.0+cu128
Batch size      : 16
Point limit     : None (full FPS point cloud)

CHECKPOINT
Model name      : DGCNNRegressor
Best epoch      : 96
Validation loss : 0